# Error Case 5 — 403 Forbidden: Wrong Privilege

**Error:** `403 Forbidden`  
**Root cause:** Principal has insufficient privilege (e.g. READ but attempting WRITE)  
**Fix:** Grant required privilege to catalog role

## Flow
```
POST /catalog/v1/{catalog}/namespaces (requires CREATE_NAMESPACE)
  → Auth: JWT valid ✅
  → RBAC: grant_records found
  → privilege check: TABLE_READ_DATA only
  → CREATE_NAMESPACE not granted → 403
```

In [4]:
import sys
sys.path.insert(0, '..')
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [5]:
# Cell 2: Full setup with READ-ONLY privilege
r1 = create_catalog()
assert r1.status_code == 201, r1.text
r2 = create_principal()
assert r2.status_code == 201
cred = r2.json()['credentials']
r3 = create_principal_role()
assert r3.status_code == 201
r4 = create_catalog_role()
assert r4.status_code == 201

# Grant READ-ONLY privilege (not CATALOG_MANAGE_CONTENT)
r5 = grant_privilege(privilege='TABLE_READ_DATA')
assert r5.status_code == 201

r6 = assign_catalog_role_to_principal_role()
assert r6.status_code == 201
r7 = assign_principal_role_to_principal()
assert r7.status_code == 201

print('Setup: full RBAC chain with TABLE_READ_DATA only')

Setup: full RBAC chain with TABLE_READ_DATA only


In [6]:
# Cell 3: Trigger 403 — attempt CREATE_NAMESPACE with READ privilege
r_tok = get_token(cred['clientId'], cred['clientSecret'])
assert r_tok.status_code == 200
principal_token = r_tok.json()['access_token']

# READ works (TABLE_READ_DATA allows listing)
r_read = requests.get(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token)
)
print(f'READ (list namespaces): {r_read.status_code} ← should be 200')

# WRITE fails (no CREATE_NAMESPACE privilege)
r_write = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token),
    json={'namespace': [TEST_NAMESPACE], 'properties': {}}
)
print_response(r_write, '403 Wrong privilege response')
assert r_write.status_code == 403
print('\n✅ 403 confirmed — insufficient privilege')

READ (list namespaces): 403 ← should be 200

403 Wrong privilege response:
  Status:     403
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000162
  Body: {
  "error": {
    "message": "Principal 'test-principal' with activated PrincipalRoles '[test-principal-role]' and activated grants via '[test-principal-role, test-catalog-role]' is not authorized for op CREATE_NAMESPACE",
    "type": "ForbiddenException",
    "code": 403
  }
}

✅ 403 confirmed — insufficient privilege


In [7]:
# Cell 4: Find log in OpenSearch
wait_for_logs(5)
request_id = get_request_id(r_write)
hits = search_and_debug(request_id, minutes_ago=10)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
🔍 Searching for requestId: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000162
✅ Found by requestId
Found 20 log entries:

ℹ️  [2026-06-02T00:58:42.830Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000162
   realmId   : POLARIS
   message   : 192.168.194.1 - test-principal [02/Jun/2026:00:58:42 +0000] "POST /api/catalog/v1/test-error-catalog/namespaces HTTP/1.1" 403 253

ℹ️  [2026-06-02T00:58:42.830Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000162
   realmId   : POLARIS
   message   : 192.168.194.1 - test-principal [02/Jun/2026:00:58:42 +0000] "POST /api/catalog/v1/test-error-catalog/namespaces HTTP/1.1" 403 253

🔍 [2026-06-02T00:58:42.829Z] DEBUG
   logger    : org.apache.polaris.service.exception.IcebergExceptionMapper
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000162
   real

In [8]:
# Cell 5: Fix — grant CATALOG_MANAGE_CONTENT
r_fix = grant_privilege(privilege='CATALOG_MANAGE_CONTENT')
print_response(r_fix, 'Grant CATALOG_MANAGE_CONTENT')
assert r_fix.status_code == 201

# Verify — now CREATE_NAMESPACE should work
r_tok = get_token(cred['clientId'], cred['clientSecret'])
principal_token = r_tok.json()['access_token']

r_verify = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(principal_token),
    json={'namespace': [TEST_NAMESPACE], 'properties': {}}
)
print(f'\nVerify create namespace: {r_verify.status_code}')
assert r_verify.status_code == 200 or r_verify.status_code == 201
print('\n✅ Fixed — WRITE now works')

cleanup()
print('\n=== Summary ===')
print('Error:    403 Forbidden')
print('Cause:    Insufficient privilege for operation')
print('Detect:   WARN message contains lacks/privilege')
print('Fix:      Grant required privilege to catalog role')


Grant CATALOG_MANAGE_CONTENT:
  Status:     201
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000164
  Body: 

Verify create namespace: 200

✅ Fixed — WRITE now works
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    403 Forbidden
Cause:    Insufficient privilege for operation
Detect:   WARN message contains lacks/privilege
Fix:      Grant required privilege to catalog role
